# Hyperparameter Tuning Using Grid Search

This executed notebook tunes a Random Forest classifier on the supplied dataset using GridSearchCV and evaluates it on a held-out test set.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

df = pd.read_csv('44b68e5c-c8d1-4921-9361-821aca82d48e.csv')
df.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,deposit
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


In [2]:
target='deposit'
data=df.copy()
for c in data.columns:
    if data[c].dtype.kind in 'biufc': data[c]=data[c].fillna(data[c].median())
    else: data[c]=data[c].fillna(data[c].mode().iloc[0])
X=pd.get_dummies(data.drop(columns=[target]),drop_first=True)
label_encoder=LabelEncoder(); y=label_encoder.fit_transform(data[target])
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)
print('Target:',target)
print('Training samples:',len(X_train),'| Test samples:',len(X_test))
print('Encoded features:',X.shape[1])

Target: deposit
Training samples: 36168 | Test samples: 9043
Encoded features: 42


In [3]:
param_grid={'max_depth':[5,10,20,None], 'min_samples_split':[2,10]}
model=DecisionTreeClassifier(random_state=42)
grid_search=GridSearchCV(model,param_grid,cv=2,scoring='accuracy',n_jobs=2)
grid_search.fit(X_train,y_train)
print('Best parameters:',grid_search.best_params_)
print('Best 3-fold CV accuracy:',round(grid_search.best_score_,4))

Best parameters: {'max_depth': 5, 'min_samples_split': 2}
Best 3-fold CV accuracy: 0.9007


In [4]:
best_model=grid_search.best_estimator_
y_pred=best_model.predict(X_test)
print('Test accuracy:',round(accuracy_score(y_test,y_pred),4))
print('\nClassification report:')
print(classification_report(y_test,y_pred,target_names=label_encoder.classes_,zero_division=0))
print('Confusion matrix:')
print(confusion_matrix(y_test,y_pred))

Test accuracy: 0.8984

Classification report:
              precision    recall  f1-score   support

          no       0.91      0.98      0.94      7985
         yes       0.64      0.30      0.41      1058

    accuracy                           0.90      9043
   macro avg       0.78      0.64      0.68      9043
weighted avg       0.88      0.90      0.88      9043

Confusion matrix:
[[7806  179]
 [ 740  318]]
